# Telco Customer Churn — EDA, Modeling, Thresholding & Explainability

This notebook is the **analysis companion** to the production repo (FastAPI + artifacts).  
It’s designed to run both:

- **Locally** (dataset at `data/Telco-Customer-Churn.csv`)
- **On Kaggle** (dataset under `/kaggle/input/...`)

## Tips
- Keep feature engineering in `src/` for consistency with the API.
- Save the production model as a **Booster JSON** (`xgb_booster.json`) for stability across XGBoost versions.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np 
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.metrics import precision_recall_curve
import xgboost as xgb


In [ ]:
df = pd.read_csv('/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()
df.info()
df.describe()

Many fields appear to be incorrectly categorized like 'TotalCharges'. Let's ensure that fields are in the appropriate format before going forward. 

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].isna().sum()

In [ ]:
df[df['TotalCharges'].isna()]


These users have missing values because they are new and have not been charged yet. We should drop these users since they are all too new to have useful behavioral patterns to help us predict churn. 

In [ ]:
df = df[df['TotalCharges'].notna()]
df.reset_index(drop=True, inplace=True)

Let's now look at the target variable Churn in more detail

In [ ]:
sns.countplot(data=df, x='Churn')
plt.title('Churn Class Distribution')
plt.show()

df['Churn'].value_counts(normalize=True)


Only moderate imbalance, but good to know when making modeling and evaluation considerations later. 

Let's now drop non-predictive columns, turn 'churn' into a binary format, and encode categorical values.. 

In [ ]:
df = df.drop(columns=['customerID'])

In [ ]:
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})


In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)
df_encoded.info()

Now let's build our first basic model

In [ ]:
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

In [ ]:

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]  # needed for AUC

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

Overall, the model performance looks pretty good for a first pass, but the class imbalance looks like it needs to be addressed. 

For both precision and recall it looks like we're not great at correctly identifying users that will churn and may be missing too many opportunities to intervene. 

Depending on the overall costs of whatever intervention the business runs, my assumption is that it makes more sense to potentially have more false positives than miss too many users at risk of churning

Let's try giving more weight to minority class (churn = 1) during training.

In [ ]:
rf_balanced = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_balanced.fit(X_train, y_train)

y_pred_bal = rf_balanced.predict(X_test)
y_prob_bal = rf_balanced.predict_proba(X_test)[:, 1]

from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred_bal))
print("ROC AUC:", roc_auc_score(y_test, y_prob_bal))

Even with class weighting, the model didn’t improve churn recall as much as hoped — probably because Random Forest already handled imbalance decently, or the signal for churn is noisy.

Let's try tweaking the probability threshold.

In [ ]:
y_pred_thresh = np.where(y_prob_bal > 0.415, 1, 0)

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_thresh))


We've caught more true churners — recall jumped by 10 points, which is significant. This is a tradeoff:

Slightly more false positives (lower precision)

But more churners identified early = more opportunity to act

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_bal)

plt.figure(figsize=(8,6))
plt.plot(thresholds, precisions[:-1], label='Precision')
plt.plot(thresholds, recalls[:-1], label='Recall')
plt.axvline(0.415, color='gray', linestyle='--', label='Current Threshold (0.415)')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision vs. Recall Tradeoff')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
xgb_model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)  # handles imbalance
)

xgb_model.fit(X_train, y_train)

In [ ]:
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_pred_xgb = (y_prob_xgb > 0.415).astype(int)  # use your new threshold

print(classification_report(y_test, y_pred_xgb))
print("ROC AUC:", roc_auc_score(y_test, y_prob_xgb))

I'm catching a lot more churners now — at the cost of more false positives, which I already expected and am willing to accept.

F1 is up, which is a good sign that I'm not overfitting or skewing too far.

Slight drop in precision is acceptable if the intervention cost is relatively low compared to the cost of missed churn. In the real world, this would be an important consideration to make and ideally up-front with stakeholders. You'd need to model the cost of intervention with the value of retaining a customer. 

Below is code to give an example of how you might discuss decisions with stakeholders and set targets for things like 'intervention success rate'.  

What you can see from the charts below is 1) the intervention success rate is critical whether precision or recall is prioritized 2) that, if the intervention success rate meets a certain minimum threshold, proiritising recall is more profitable



In [ ]:
def churn_intervention_roi(
    n_customers=10000,
    churn_rate=0.25,
    precision=0.5,
    recall=0.75,
    intervention_cost=10,
    retained_value=200,
    success_rates=[0.1, 0.3, 0.5]
):
    churners = n_customers * churn_rate
    true_positives = churners * recall
    predicted_churn = true_positives / precision
    cost = predicted_churn * intervention_cost

    print(f"Churners: {int(churners)} | Flagged: {int(predicted_churn)}")
    print(f"Intervention Cost: £{cost:,.2f}")

    for success in success_rates:
        retained = true_positives * success
        value = retained * retained_value
        net = value - cost
        print(f"\n🎯 Success Rate: {int(success*100)}%")
        print(f"  Retained Customers: {int(retained)}")
        print(f"  Value Generated: £{value:,.2f}")
        print(f"  Net Benefit: £{net:,.2f}")

# Example use:
churn_intervention_roi()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Define function to calculate net benefit
def calculate_net_benefit(churn_rate, recall_rate, precision_rate, intervention_cost, success_rate, customer_value):
    total_customers = 10000
    churners = total_customers * churn_rate
    true_positives = churners * recall_rate
    flagged_customers = true_positives / precision_rate
    cost = flagged_customers * intervention_cost
    customers_saved = true_positives * success_rate
    value_generated = customers_saved * customer_value
    net_benefit = value_generated - cost
    return net_benefit

# Define parameters
churn_rate = 0.25
intervention_cost = 10  # £ per intervention
customer_value = 200  # £ per retained customer

# Success rate scenarios (10%, 30%, 50%)
success_rates = [0.10, 0.30, 0.50]
models = ['Model A', 'Model B']

# Dataframe to store results
results = []

# Model A (High Precision: 70% Precision, 40% Recall)
for success_rate in success_rates:
    net_benefit = calculate_net_benefit(churn_rate, 0.40, 0.70, intervention_cost, success_rate, customer_value)
    results.append(['Model A', success_rate, net_benefit])

# Model B (High Recall: 75% Recall, 50% Precision)
for success_rate in success_rates:
    net_benefit = calculate_net_benefit(churn_rate, 0.75, 0.50, intervention_cost, success_rate, customer_value)
    results.append(['Model B', success_rate, net_benefit])

# Convert results to DataFrame
df_results = pd.DataFrame(results, columns=['Model', 'Success Rate', 'Net Benefit'])

# Plotting the results
plt.figure(figsize=(10,6))
for model in models:
    model_data = df_results[df_results['Model'] == model]
    plt.plot(model_data['Success Rate'], model_data['Net Benefit'], marker='o', label=model)

plt.title('Net Benefit vs Intervention Success Rate (Model A vs Model B)')
plt.xlabel('Intervention Success Rate')
plt.ylabel('Net Benefit (£)')
plt.legend()
plt.grid(True)
plt.show()


We haven't done much EDA or feature engineering yet, before trying any other models or hyperparameter tuning, let's see if we can get value from our features. 

In [ ]:
# Compute correlation matrix
corr = df_encoded.corr()

# Focus on how features relate to churn
churn_corr = corr['Churn'].drop('Churn').sort_values(ascending=False)

# Plot top 10 positively and negatively correlated features
top_pos = churn_corr.head(10)
top_neg = churn_corr.tail(10)

plt.figure(figsize=(10,6))
sns.barplot(x=top_pos.values, y=top_pos.index)
plt.title("Top Positive Correlations with Churn")
plt.show()

plt.figure(figsize=(10,6))
sns.barplot(x=top_neg.values, y=top_neg.index)
plt.title("Top Negative Correlations with Churn")
plt.show()


In [ ]:
churn_corr = df_encoded.corr()['Churn'].drop('Churn').sort_values(ascending=False)
churn_corr


Fiber optic users may be higher-cost or more demanding

Electronic check may be linked to lower tech-savviness or payment issues

High monthly charges are unsurprisingly correlated with higher churn

Longevity and contract type strongly reduce churn

Customers with tech support and dependents are more loyal

let's try engineering some new features

In [ ]:
df_encoded['total_value'] = df_encoded['tenure'] * df_encoded['MonthlyCharges']
df_encoded['avg_monthly_charge'] = df_encoded['TotalCharges'] / df_encoded['tenure'].replace(0, 1)  # avoid divide-by-zero
service_cols = [
    'OnlineSecurity_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes',
    'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes'
]
df_encoded['num_services'] = df_encoded[service_cols].sum(axis=1)
df_encoded['tenure_bucket'] = pd.cut(df_encoded['tenure'], bins=[0, 12, 24, 48, 72], labels=False)
df_encoded['is_month_to_month'] = (~(df_encoded['Contract_One year'] | df_encoded['Contract_Two year'])).astype(int)


In [ ]:
engineered = ['total_value', 'avg_monthly_charge', 'num_services', 'tenure_bucket', 'is_month_to_month']
df_encoded[engineered + ['Churn']].corr()['Churn'].sort_values(ascending=False)

In [ ]:
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
xgb_model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)  # handles imbalance
)

xgb_model.fit(X_train, y_train)

In [ ]:

y_prob = xgb_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.415).astype(int)

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
import xgboost as xgb

xgb_clf = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)  # keep this fixed
)

param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.2),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
}

In [ ]:
random_search = RandomizedSearchCV(
    xgb_clf,
    param_distributions=param_dist,
    n_iter=25,  # Reduce for speed, increase for thoroughness
    scoring='roc_auc',
    cv=3,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

In [ ]:
best_model = random_search.best_estimator_

y_prob_best = best_model.predict_proba(X_test)[:, 1]
y_pred_best = (y_prob_best > 0.415).astype(int)

from sklearn.metrics import classification_report, roc_auc_score
print("Best Params:", random_search.best_params_)
print(classification_report(y_test, y_pred_best))
print("ROC AUC:", roc_auc_score(y_test, y_prob_best))

In [ ]:
import shap
import matplotlib.pyplot as plt

In [ ]:
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

In [ ]:
shap.summary_plot(shap_values, X_test)

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type="bar")

Tldr; Churn risk is structurally driven by contract type and lifecycle stage, with pricing sensitivity and service engagement acting as secondary amplifiers.

Insights: 

Contract structure (particularly month-to-month vs. fixed-term) is the dominant structural driver of churn risk.

Churn risk is disproportionately concentrated in early lifecycle customers (low tenure), indicating onboarding and early value realization are critical.

While higher monthly charges increase churn propensity, additional service adoption appears to mitigate this effect — suggesting that engagement depth offsets pricing sensitivity.

Recommendations: 

Strategically focus on early-stage efforts in a customer's lifecycle to shift from M2M contracts to longer-term ones as well as onboard them to other services provided such as online security. Test strategies such as offering early trials, discounts, or other incentives to shift users onto these contracts and products.

Test ideas:

Limited-time contract upgrade incentives

Contract bundling with additional services

Discounted 1-year commitments during onboarding window

Follow-up deep-dive work could 1) identify and capture additional features to add to the model 2) examine churn prediction and reduction strategies for longer-tenured customers specifically as the drivers and interventions would likely be different for this cohort. 

In [ ]:
print("Best Parameters:", random_search.best_params_)
print("\nFinal Evaluation at Threshold = 0.415")
print(classification_report(y_test, y_pred_best))
print("ROC AUC:", roc_auc_score(y_test, y_prob_best))

Final model: XGBoost with tuned hyperparameters
ROC AUC: X
Churn recall at 0.415 threshold: 0.82
Precision: 0.47
Business threshold selected based on ROI modeling

In [ ]:
import json
from pathlib import Path
import joblib
import xgboost as xgb

ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Save model in a stable format (recommended)
best_model.get_booster().save_model(str(ARTIFACTS_DIR / "xgb_booster.json"))

# Save threshold + feature columns for inference alignment
FINAL_THRESHOLD = 0.415
joblib.dump(FINAL_THRESHOLD, ARTIFACTS_DIR / "decision_threshold.pkl")
joblib.dump(list(X_train.columns), ARTIFACTS_DIR / "feature_columns.pkl")

metadata = {
    "xgboost": xgb.__version__,
    "threshold": float(FINAL_THRESHOLD),
}
(ARTIFACTS_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))

print("Saved artifacts to artifacts/: xgb_booster.json, decision_threshold.pkl, feature_columns.pkl, metadata.json")


In [ ]:
threshold = 0.415
joblib.dump(threshold, "decision_threshold.pkl")

In [ ]:
import json
from pathlib import Path
import joblib
import xgboost as xgb

ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Save model in a stable format (recommended)
best_model.get_booster().save_model(str(ARTIFACTS_DIR / "xgb_booster.json"))

# Save threshold + feature columns for inference alignment
FINAL_THRESHOLD = 0.415
joblib.dump(FINAL_THRESHOLD, ARTIFACTS_DIR / "decision_threshold.pkl")
joblib.dump(list(X_train.columns), ARTIFACTS_DIR / "feature_columns.pkl")

metadata = {
    "xgboost": xgb.__version__,
    "threshold": float(FINAL_THRESHOLD),
}
(ARTIFACTS_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))

print("Saved artifacts to artifacts/: xgb_booster.json, decision_threshold.pkl, feature_columns.pkl, metadata.json")


In [ ]:
example_row = X_test.iloc[[0]].copy()
example_row.to_json("example_input.json", orient="records")